In [ ]:
# Connect to DB

# Must run in terminal while open (check keypass): ssh ...

using LibPQ
using DataFrames

# Connect to the AWS RDS instance via local SSH tunnel
conn_str = get(ENV, "PG_CONN", "")
if isempty(strip(conn_str))
    host = get(ENV, "PGHOST", "")
    port = get(ENV, "PGPORT", "")
    user = get(ENV, "PGUSER", "")
    dbname = get(ENV, "PGDATABASE", "")
    password = get(ENV, "PGPASSWORD", "")
    conn_str = "host=$host port=$port user=$user dbname=$dbname password=$password"
end
conn = LibPQ.Connection(conn_str)

PostgreSQL connection (CONNECTION_OK) with parameters:
  user = testnikoloda
  password = ********************
  channel_binding = prefer
  dbname = postgres
  host = localhost
  port = 5433
  client_encoding = UTF8
  options = -c DateStyle=ISO,YMD -c IntervalStyle=iso_8601 -c TimeZone=UTC
  application_name = LibPQ.jl
  sslmode = prefer
  sslcompression = 0
  sslcertmode = allow
  sslsni = 1
  ssl_min_protocol_version = TLSv1.2
  gssencmode = disable
  krbsrvname = postgres
  gssdelegation = 0
  target_session_attrs = any
  load_balance_hosts = disable

In [18]:
# Create Tables

# 1. Subscribers
# Using standard INTEGER since you manually provide the IDs 1-5
execute(conn, """
CREATE TABLE IF NOT EXISTS subscribers(
    account_id INTEGER PRIMARY KEY,
    name TEXT, 
    email TEXT
);
""")

# 2. Buses
# Added UNIQUE to bus_id so the records table can reference it as a foreign key
execute(conn, """
CREATE TABLE IF NOT EXISTS buses(
    bus_key_id SERIAL PRIMARY KEY,
    bus_id INTEGER UNIQUE, 
    address TEXT,
    account_id INTEGER REFERENCES subscribers(account_id)
);
""")

# 3. Records
# Using SERIAL for auto-incrementing primary keys, and TIMESTAMP instead of DATETIME
execute(conn, """
CREATE TABLE IF NOT EXISTS records(
    record_id SERIAL PRIMARY KEY,
    record_time TIMESTAMP, 
    power_quality REAL,
    status INTEGER,
    bus_id INTEGER REFERENCES buses(bus_id)
);
""")

# 4. Global Records
execute(conn, """
CREATE TABLE IF NOT EXISTS globalRecords(
    global_record_id SERIAL PRIMARY KEY,
    record_time TIMESTAMP, 
    power_quality REAL,
    num_islands INTEGER,
    converges INTEGER
);
""")

NOTICE:  relation "subscribers" already exists, skipping
NOTICE:  relation "records" already exists, skipping
NOTICE:  relation "globalrecords" already exists, skipping


PostgreSQL result

In [19]:
# Populate subscribers and buses
subs = [
    (1, "Alice Vance", "alice@grid.org"),
    (2, "Bob Miller", "bob@outlook.com"),
    (3, "Charlie Day", "charlie@paddys.pub"),
    (4, "Diana Prince", "diana@gmail.com"),
    (5, "Edward Jones", "ed@riddle.co")
]

# LibPQ uses $1, $2 positional parameters instead of ?
for s in subs
    execute(conn, "INSERT INTO subscribers (account_id, name, email) VALUES (\$1, \$2, \$3) ON CONFLICT DO NOTHING", [s[1], s[2], s[3]])
end

buses = [
    (1, 1, "Sector 7G Power Plant", 1),
    (2, 2, "North Substation A", 1),
    (3, 3, "Downtown Hub", 2),
    (4, 4, "South Side Transformer", 3),
    (5, 5, "West End Feeder", 3),
    (6, 6, "Charlie's Home", 3),
    (7, 7, "Amazonian Relay", 4),
    (8, 8, "Question Mark Manor", 5)
]

for b in buses
    execute(conn, "INSERT INTO buses (bus_key_id, bus_id, address, account_id) VALUES (\$1, \$2, \$3, \$4) ON CONFLICT DO NOTHING", [b[1], b[2], b[3], b[4]])
end

println("Tables created and populated successfully!")

Tables created and populated successfully!


In [20]:
# Print
display(DataFrame(execute(conn, "SELECT * FROM buses LIMIT 5;")))

close(conn)

Row,bus_key_id,bus_id,address,account_id
,Int32?,Int32?,String?,Int32?
1,1,1,Sector 7G Power Plant,1
2,2,2,North Substation A,1
3,3,3,Downtown Hub,2
4,4,4,South Side Transformer,3
5,5,5,West End Feeder,3


In [16]:
# This will delete the buses table and any foreign key constraints tied to it
execute(conn, "DROP TABLE IF EXISTS buses CASCADE;")

NOTICE:  drop cascades to constraint records_bus_id_fkey on table records


PostgreSQL result